In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import textattack
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
from langdetect import detect

c:\Users\Shaz\interp-toxicity\interp-toxicity\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from tqdm import tqdm, trange

from datasets import load_dataset
import pandas as pd
import functools
import sys
from pathlib import Path
from typing import Callable

# import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import eindex
# from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
# from transformer_lens import (
#     ActivationCache,
#     FactoredMatrix,
#     HookedTransformer,
#     HookedTransformerConfig,
#     HookedEncoderDecoder,
#     HookedEncoder,
#     utils,
# )
# from transformer_lens.hook_points import HookPoint

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
# from transformer_lens import HookedTransformer
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
tqdm.pandas()

In [3]:
print("Available CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

Available CUDA devices: 1
Device 0: NVIDIA GeForce RTX 3060 Laptop GPU


## Baseline

In [14]:
df = pd.read_csv('jigsaw/test_clean.csv')

In [6]:
tokenizer = AutoTokenizer.from_pretrained("s-nlp/roberta_toxicity_classifier")
model = AutoModelForSequenceClassification.from_pretrained("s-nlp/roberta_toxicity_classifier")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\Shaz\interp-toxicity\interp-toxicity\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shaz\.cache\huggingface\hub\models--s-nlp--roberta_toxicity_classifier. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [16]:
from sklearn.metrics import accuracy_score
from tqdm import trange

# Batch size for GPU inference
BATCH_SIZE = 64

all_preds = []
all_labels = []

model = model.to(device)
model.eval()
new_df = df.sample(5000)
num_samples = len(new_df)
for start_idx in trange(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    batch_texts = new_df.comment_text.iloc[start_idx:end_idx].tolist()
    batch_labels = new_df.toxic.iloc[start_idx:end_idx].astype(int).tolist()
    inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch_labels)

accuracy = accuracy_score(all_labels, all_preds)
print(f"Baseline model accuracy on test set: {accuracy:.4f}")

100%|██████████| 79/79 [01:31<00:00,  1.16s/it]

Baseline model accuracy on test set: 0.9326


In [22]:
# Import our PGD attack implementation
from pgd_bert_attack import PGDBERTAttack, set_seed

# Set seed for reproducibility
set_seed(42)

In [23]:
# Initialize PGD Attack
print("Setting up PGD BERT Attack...")

# Create the attacker - it will automatically load BERT MLM model for candidate generation
attacker = PGDBERTAttack(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_iters=10,
    top_k_tokens=5,
    mlm_top_k=50,
    sim_threshold=0.8,  # Slightly lower threshold for more flexibility
    max_length=512
)

print("PGD Attack setup complete!")


Setting up PGD BERT Attack...
Loading BERT MLM model for candidate generation...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


PGD Attack setup complete!


In [25]:
# Evaluate model robustness on a sample of data
print("Evaluating model robustness with PGD attacks...")

# Sample some data for evaluation
sample_size = 100  # Reduced for demonstration
sample_df = df.sample(sample_size, random_state=42)
sample_texts = sample_df.comment_text.tolist()
sample_labels = sample_df.toxic.astype(int).tolist()

print(f"Running attacks on {sample_size} samples...")
print(f"Sample distribution: {sum(sample_labels)} toxic, {len(sample_labels) - sum(sample_labels)} non-toxic")

# Run batch attack evaluation with saving enabled
output_file = "snlp_roberta/jigsaw_pgd.jsonl"
robustness_results = attacker.evaluate_robustness(
    sample_texts, 
    sample_labels,
    output_file=output_file,
    log_interval=100  # Save every 25 examples
)

print(f"\nRobustness Evaluation Results:")
print(f"Total samples: {robustness_results['total_samples']}")
print(f"Successful attacks: {robustness_results['successful_attacks']}")
print(f"Attack success rate: {robustness_results['attack_success_rate']:.2%}")
print(f"Average iterations for successful attacks: {robustness_results['average_iterations']:.1f}")
print(f"Results saved to: {output_file}")


Evaluating model robustness with PGD attacks...
Running attacks on 5000 samples...
Sample distribution: 433 toxic, 4567 non-toxic
Evaluating robustness on 5000 samples...


Attacking:   0%|          | 10/5000 [02:32<24:49:50, 17.91s/it]

Error getting MLM candidates: index 0 is out of bounds for dimension 0 with size 0
Processed 10/5000, Success rate: 60.00%


Attacking:   0%|          | 14/5000 [03:59<23:34:16, 17.02s/it]

Error getting MLM candidates: index 0 is out of bounds for dimension 0 with size 0
Error getting MLM candidates: index 0 is out of bounds for dimension 0 with size 0


...

Processed 20/5000, Success rate: 60.00%


Attacking:   1%|          | 30/5000 [08:35<22:37:23, 16.39s/it]

Processed 30/5000, Success rate: 60.00%


Attacking:   1%|          | 32/5000 [08:50<22:53:02, 16.58s/it]


KeyboardInterrupt: 

In [ ]:
# Analyze attack results and show examples
print("Analysis of Attack Results:")
print("=" * 50)

results = robustness_results['results']
successful_attacks = [(i, adv_text, meta) for i, (adv_text, meta) in enumerate(results) if meta['success']]

if successful_attacks:
    print(f"\nShowing first 5 successful attacks:")
    print("-" * 40)
    
    for i, (idx, adv_text, meta) in enumerate(successful_attacks[:5]):
        original_text = meta['original_text']
        print(f"\nExample {i+1}:")
        print(f"Original: {original_text}")
        print(f"Adversarial: {adv_text}")
        print(f"Original label: {sample_labels[idx]}")
        print(f"Iterations: {meta['iters']}")
        
        # Check the difference
        from difflib import SequenceMatcher
        similarity = SequenceMatcher(None, original_text, adv_text).ratio()
        print(f"Text similarity: {similarity:.3f}")
        print("-" * 40)
else:
    print("No successful attacks found in this sample.")

# Attack success by label
print(f"\nAttack Success by Original Label:")
toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 1]
non_toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 0]

if toxic_attacks:
    toxic_success_rate = sum(1 for _, meta in toxic_attacks if meta['success']) / len(toxic_attacks)
    print(f"Toxic examples: {toxic_success_rate:.2%} success rate ({len(toxic_attacks)} samples)")

if non_toxic_attacks:
    non_toxic_success_rate = sum(1 for _, meta in non_toxic_attacks if meta['success']) / len(non_toxic_attacks)
    print(f"Non-toxic examples: {non_toxic_success_rate:.2%} success rate ({len(non_toxic_attacks)} samples)")

print(f"\nOverall model robustness: {100 - robustness_results['attack_success_rate']*100:.1f}% robust to PGD attacks")


In [ ]:
# Demonstrate loading and analyzing saved results
print("Loading and analyzing saved attack results...")

# You can load results from a previous run
if output_file:
    # Analyze the saved results
    analysis = attacker.analyze_saved_results(output_file)
    
    print(f"\nDetailed breakdown:")
    print(f"- Toxic examples attacked: {analysis.get('toxic_samples', 0)}")
    print(f"- Non-toxic examples attacked: {analysis.get('non_toxic_samples', 0)}")
    print(f"- Robustness score: {100 - analysis.get('attack_success_rate', 0)*100:.1f}%")
    
    # You can also manually load the raw data for custom analysis
    raw_results = attacker.load_saved_results(output_file)
    print(f"\nFirst saved result example:")
    if raw_results:
        first_result = raw_results[0]
        print(f"ID: {first_result.get('id')}")
        print(f"Original: {first_result.get('orig_text', '')[:100]}...")
        print(f"Adversarial: {first_result.get('adv_text', '')[:100]}...")
        print(f"Success: {first_result.get('success')}")
        print(f"Iterations: {first_result.get('iters')}")
else:
    print("No output file specified - skipping analysis demo")


## Attack